In [ ]:
import pandas as pd
import numpy as np
import re

In [ ]:
# tabela = pd.read_excel()

In [ ]:
meses = { # os dois padrões de datas que vem em strings
        "janeiro": "January", "fevereiro": "February", "março": "March",
        "abril": "April", "maio": "May", "junho": "June", "julho": "July",
        "agosto": "August", "setembro": "September", "outubro": "October",
        "novembro": "November", "dezembro": "December",
        "jan": "January", "fev": "February", "mar": "March",
        "abr": "April", "mai": "May", "jun": "June", "jul": "July",
        "ago": "August", "set": "September", "out": "October",
        "nov": "November", "dez": "December"
    }

def ajustar_espacos(datas): # função pra simplificar, basicamente se tiver espaços repetidos ele retorna apenas 1 espaço
    return [re.sub(r"\s+", " ", s.strip()) if isinstance(s, str) else s for s in datas]

def traduz_mes(s):
    if isinstance(s, str):
        s = s.replace("-", " ").lower()
        pattern = re.compile("|".join(map(re.escape, meses.keys())))
        s = pattern.sub(lambda m: meses[m.group()], s)
    return s

def converter_data(array):
    array = pd.Series(ajustar_espacos(array))
    arr_dt = array.apply(traduz_mes)

    result = pd.to_datetime(arr_dt, format="%B %Y", errors='coerce')
    # converte o %Y

    # os que não foram convertidos como %Y vai como %y
    mask = result.isna()
    if mask.any():
        result[mask] = pd.to_datetime(arr_dt[mask], format="%B %y", errors='coerce')
    return result

In [ ]:
teste = pd.Series([
    "jan 23", "fev 23", "mar 23", "abr 23", "mai 23", "jun 23",
    "jul 23", "ago 23", "set 23", "out 23", "nov 23", "dez 23",
    "janeiro 2024", "fevereiro 2024", "março 2024", "abril 2024",
    "maio 2024", "junho 2024", "julho 2024", "agosto 2024",
    "setembro 2024", "outubro 2024", "novembro 2024", "dezembro 2024"])

In [ ]:
teste

In [ ]:
converter_data(teste) # funcionando OK aparentemente

In [ ]:
"""def limpar_linhas_extras(df):
    #ajeitar para não apagar a primeira coluna "Item"
    def is_date(value) or :
        try:
            pd.to_datetime(value)
            return True
        except:
            return False

    primeira_linha = df.iloc[0]
    colunas_para_manter = [col for col in df.columns if is_date(primeira_linha[col])]
    return df[colunas_para_manter]"""

In [ ]:
def limpar_colunas_extras(df):
    cols_to_drop_idx = []

    for i, col in enumerate(df.columns):
        value = df.iloc[0, i]
        if pd.isna(value):
            cols_to_drop_idx.append(i)
        else:
            try:
                pd.to_datetime(value, errors='raise')
            except (ValueError, TypeError):
                cols_to_drop_idx.append(i)

    # lista dos índices das colunas a manter
    cols_to_keep_idx = [i for i in range(len(df.columns)) if i not in cols_to_drop_idx]

    # seleciona só as colunas que queremos manter
    df_cleaned = df.iloc[:, cols_to_keep_idx]

    return df_cleaned

In [ ]:
teste2 = pd.read_excel(r"C:\Users\murilo.pinheiro\OneDrive - V8 Capital\Desktop\FIDC INVESTIDORES\ALFA FIDC - ABRIL 2025 INVESTIDORES\ALFA FIDC - Dados Basicos - Informações de Investimentos - Abril2025.xlsx")

In [ ]:
random = teste2.iloc[:, 0]

random

In [ ]:
teste2

In [ ]:
limpar_colunas_extras(teste2) # testar melhor depois

In [ ]:
teste3 = pd.read_excel(r"C:\Users\murilo.pinheiro\OneDrive - V8 Capital\Desktop\FIDC INVESTIDORES\AURUM FIDC - ABRIL 2025 INVESTIDORES\AURUM FIDC - Informação aos Investidores - Abril 2025.xlsx")

In [ ]:
teste3.iloc[:, 1]

In [ ]:
random = teste3.iloc[:, 0]

In [ ]:
import re

# Padrões com \d+ para representar qualquer número
#Padronizar os dias
def verificar_padrao(entry):
    text = str(entry)
    padroes = [
    r"^\d+\s*a\s*\d+$",
    r"^\d+\s*-\s*\d+$",
    r"^\d+\s*-\s*\d+\s*dias$",
    r"^de\s+\d+\s*a\s+\d+\s*dias$",
    r"^\d+\s*e\s*\d+\s*dias$",
    r"^acima\s+de\s+\d+\s*dias$",
    r"^superior\s+a\s+\d+$",
    r"^superior\s+\d+$",
    r"^>\s*\d+$",
    r"^até\s+\d+$",
    r"^até\s+\d+\s*dias$"
    ]
    text = text.lower().strip()
    return any(re.search(padrao, text) for padrao in padroes)

def padronizar_string(texto):
    if not texto or not isinstance(texto, str):
        return None

    texto = texto.lower().strip()
    subtrair_se_comecar_em = {6, 16, 31, 61, 91, 121, 151, 181, 366, 721}

    padroes = [
        (r"^1-(\d+)", lambda m: f"0-{m.group(1)} dias"),

        (r"^de\s*(\d+)\s*a\s*(\d+)\s*dias?$", lambda m:
         f"{int(m.group(1)) - 1 if int(m.group(1)) in subtrair_se_comecar_em else m.group(1)}-{m.group(2)} dias"),

        (r"^(\d+)\s*(?:a|e|-)\s*(\d+)\s*dias?$", lambda m:
         f"{int(m.group(1)) - 1 if int(m.group(1)) in subtrair_se_comecar_em else m.group(1)}-{m.group(2)} dias"),

        (r"^(\d+)\s*(?:a|e|-)\s*(\d+)$", lambda m:
         f"{int(m.group(1)) - 1 if int(m.group(1)) in subtrair_se_comecar_em else m.group(1)}-{m.group(2)} dias"),

        (r"^até\s*(\d+)(?:\s*dias?)?$", lambda m: f"<= {m.group(1)} dias"),

        (r"^>\s*(\d+)$", lambda m: f"> {int(m.group(1))-1 if int(m.group(1)) == 121 else m.group(1)} dias"),

        (r"^(?:acima\s*de|superior(?:\s*a)?)\s*(\d+)(?:\s*dias?)?$", lambda m: f"> {int(m.group(1))-1 if int(m.group(1)) == 121 else m.group(1)} dias"),
    ]

    for padrao, formato in padroes:
        match = re.fullmatch(padrao, texto)
        if match:
            return formato(match)
    return None


In [ ]:
def padronizar_colunas_dias(df):
    columns = pd.Series(df.columns)

    result = columns.apply(verificar_padrao)
    days_columns = columns[result]

    final = days_columns.apply(padronizar_string)

    columns[result] = final

    df.columns = columns

    return df

In [ ]:
import yaml
def carregar_yaml(caminho_yaml):
    with open(caminho_yaml, 'r', encoding='utf-8') as f:
        dados = yaml.safe_load(f)

    return dados  # Retorna como dicionário: {coluna_final: [possibilidades]}

colunas_equivalentes = carregar_yaml("./colunas.YAML")
padroes_regex_col = carregar_yaml("./regex.YAML")

colunas_equivalentes

In [ ]:
def renomear_colunas_equivalentes(df, colunas_equivalentes_dict):
    renomear_dict = {}
    for nome_padrao, alternativas in colunas_equivalentes_dict.items():
        if alternativas is None: continue
        for nome in alternativas:
            if nome in df.columns:
                renomear_dict[nome] = nome_padrao
    return df.rename(columns=renomear_dict)

In [ ]:
def verificar_padrao2(entry):
    text = str(entry).lower().strip()
    padroes = [
        r'(\d+)\s*-\s*(\d+)\s*dias?',                   # intervalo: 10 - 20 dias
        r'>\s*(\d+)\s*dias?',                           # > 120 dias
        r'<=\s*(\d+)\s*dias?'                           # <= 10 dias
    ]
    return any(re.search(p, text) for p in padroes)

def padronizar_colunas_dias2(df):
    columns = pd.Series(df.columns)
    result = columns.apply(verificar_padrao2)  # vetor booleano, ex: [True, False, False, ...]
    last_true_idx = None
    new_columns = []

    for idx, col_name in enumerate(columns):
        if not result.iloc[idx]:
            new_columns.append(col_name)
            last_true_idx = idx
        else:
            if last_true_idx is not None:
                new_name = f"({columns.iloc[last_true_idx]}){col_name}"
                new_columns.append(new_name)
            else:
                new_columns.append(col_name)
    df.columns = new_columns
    return df

In [ ]:
import os

# Caminho da pasta com os arquivos CSV
caminho_pasta = './out'

# Dicionário para armazenar os DataFrames
csv_dict = {}

def remover_sufixos(colunas):
    colunas_novas = []
    for col in colunas:
        # Remove sufixo do tipo .1, .2, .3 no final da string
        col_nova = re.sub(r'\.\d+$', '', col)
        colunas_novas.append(col_nova)
    return colunas_novas

# Percorre todos os arquivos na pasta
for arquivo in os.listdir(caminho_pasta):
    if arquivo.endswith('.csv'):
        caminho_arquivo = os.path.join(caminho_pasta, arquivo)
        nome_arquivo = os.path.splitext(arquivo)[0]  # Remove a extensão
        df = pd.read_csv(caminho_arquivo, sep=';', encoding='utf-8-sig', index_col = "Data")  # ajuste encoding se necessário
        print(nome_arquivo)
        print(df.columns)
        df.columns = remover_sufixos(df.columns)
        df = padronizar_colunas_dias(df)
        df = renomear_colunas_equivalentes(df, colunas_equivalentes)
        df = padronizar_colunas_dias2(df)
        csv_dict[nome_arquivo] = df

In [ ]:
from collections import Counter

contador_colunas = Counter()

for df in csv_dict.values():
    contador_colunas.update(df.columns.tolist())

colunas_ordenadas = dict(
    sorted(
        contador_colunas.items(),
        key=lambda x: (x[0] is None, str(x[0]))
    )
)

print(len(colunas_ordenadas))
for coluna, qtd in colunas_ordenadas.items():
    print(f"{coluna}: {qtd}")

In [ ]:
colunas_ordenadas = dict(
    sorted(
        contador_colunas.items(),
        key=lambda x: x[1],  # ordenar pelo valor (quantidade)
        reverse=True         # maior para menor
    )
)

colunas_ordenadas

In [ ]:
for key in csv_dict.keys():
    print(key)
    print(csv_dict[key].columns)

In [ ]:
colunas_equivalentes.keys()

In [ ]:
def selecionar_colunas_por_nome(dataframes, padroes_colunas):
    # Usa apenas as chaves como nomes exatos de colunas a buscar
    nomes_colunas_procuradas = list(padroes_colunas.keys())
    print("Liquidado Total(R$)" in padroes_colunas)
    print(nomes_colunas_procuradas)
    resultado = {}

    for nome_df, df in dataframes.items():
        colunas_encontradas = []
        for col in df.columns:
            if col in nomes_colunas_procuradas:
                colunas_encontradas.append(col)
        resultado[nome_df] = colunas_encontradas

    return resultado

def selecionar_colunas_por_regex(dataframes, padroes_colunas):
    teste = list(padroes_colunas.values())[0]

    padroes_compilados = [re.compile(p) for p in teste]

    resultado = {}

    for nome_df, df in dataframes.items():
        colunas_capturadas = []
        for col in df.columns:
            for padrao in padroes_compilados:
                if padrao.fullmatch(str(col)):
                    colunas_capturadas.append(col)
                    break
        resultado[nome_df] = colunas_capturadas

    return resultado

list1 = selecionar_colunas_por_nome(csv_dict, colunas_equivalentes)
list2 = selecionar_colunas_por_regex(csv_dict, padroes_regex_col)

In [ ]:
#resultado = list1
resultado = {k: list1.get(k, []) + list2.get(k, []) for k in list1.keys()}

resultado

In [ ]:
colunas_equivalentes.keys()

In [ ]:
def selecionando_colunas_final(csv_dict, colunas):
    for key in csv_dict.keys():
        csv_dict[key] = csv_dict[key][colunas[key]]
    return csv_dict

print(resultado)
teste_final = selecionando_colunas_final(csv_dict, resultado)

In [ ]:
def juntar_por_data(csv_dict, data):
    lista_dados = []

    for key, df in csv_dict.items():
        if data in df.index:
            linha = df.loc[data]
            linha_df = pd.DataFrame([linha], index=[key])
        else:
            linha_df = pd.DataFrame([{col: pd.NA for col in df.columns}], index=[key])

        lista_dados.append(linha_df)

    resultado = pd.concat(lista_dados, axis=0, sort=True)
    return resultado

In [ ]:
# checar dps esse Taxa Média Recebíveis Média ponderada

#o padrão vai ser "Prazo Médio"
# checar dps esse Prazo Médio Recebíveis Estoque (dias úteis) e Prazo médio aquisição no mês
# formatar todas de dias corridos para úteis, se for o caso
# oq não tem especificando é corrido

#remover recebíveis

# hoje a tarde fazer:
    # ajeitar TUDO do SOLAR
    # ajeitar todas as %
    # adicionar últimos dados gerais
    # ai dps disso eu vejo mais oq adicionar e também os q ficarão no fim do DF final
    # também tenho que lembrar que não dá pra unir tudo de uma vez então pensar em como fica a estrutura final
    #    - vou manter os dataframes PARSEADOS em CSV padronizados salvos separados
    #    - manter o planilhão por mês
    # ajeitar os 0-15 que importam, colocando os padrãozinho lá

In [ ]:
teste_final["appaloosa"]

In [ ]:
def padronizar_indices_para_fim_do_mes(csv_dict):
    csv_dict_padronizado = {}

    for key, df in csv_dict.items():
        df = df.copy()

        # Garante que o índice está no formato datetime
        df.index = pd.to_datetime(df.index, errors='coerce')

        # Remove linhas com índice inválido (caso existam)
        df = df[~df.index.isna()]

        # Ajusta cada data para o último dia do mês
        df.index = df.index.to_period('M').to_timestamp('M')

        # Se houver duplicação após essa conversão, agrupar (aqui usamos média, mas pode ser alterado)
        df = df.groupby(df.index).first()

        csv_dict_padronizado[key] = df

    return csv_dict_padronizado

teste_final = padronizar_indices_para_fim_do_mes(teste_final)

In [ ]:
teste_final["solar"]

In [ ]:
resultado_quasefinal = juntar_por_data(teste_final, "2025-02-28")

In [ ]:
def mostrar_colunas_repetidas(csv_dict):
    for key, df in csv_dict.items():
        colunas_repetidas = df.columns[df.columns.duplicated()]
        if len(colunas_repetidas) > 0:
            print(f'DataFrame "{key}" tem colunas repetidas:')
            print(colunas_repetidas.tolist())
        else:
            print(f'DataFrame "{key}" não tem colunas repetidas.')

# Exemplo de uso:
mostrar_colunas_repetidas(teste_final)

In [ ]:
teste_final.keys()

In [ ]:
colunas_equivalentes.keys()

In [ ]:
from itertools import chain

lista_unica = pd.Series(chain.from_iterable(list2.values()))

testeee = list(lista_unica.drop_duplicates())

In [ ]:
def extrair_ordem(item):
    match = re.match(r'^\((.*?)\)\s*(>?=?\s*\d+)(?:\s*-\s*(\d+))?', item)
    if match:
        grupo = match.group(1)
        ini_raw = match.group(2)
        fim_raw = match.group(3)

        ini = int(re.sub(r'\D', '', ini_raw)) if ini_raw else 0
        fim = int(fim_raw) if fim_raw else (ini if '>' not in ini_raw else 9999)

        return (grupo, ini, fim)
    else:
        return ('ZZZ', 99999, 99999)  # Ficam no fim: 'Cedente', 'Sacado' etc.

In [ ]:
# Ordenar
#lista_ordenada = sorted(testeee, key=extrair_ordem)

#lista_ordenada

In [ ]:
lista_unica2 = list(colunas_equivalentes.keys())

#ordem_final = lista_unica2 +lista_ordenada

In [ ]:
#ordem_final
# resolver problemas das colunas repetidas

In [ ]:
import re

def organizar_colunas_df(lista_unica2, testeee):
    def extrair_ordem(item):
        match = re.match(r'^\((.*?)\)\s*(>?=?\s*\d+)(?:\s*-\s*(\d+))?', item)
        if match:
            grupo = match.group(1)
            ini_raw = match.group(2)
            fim_raw = match.group(3)

            ini = int(re.sub(r'\D', '', ini_raw)) if ini_raw else 0
            fim = int(fim_raw) if fim_raw else (ini if '>' not in ini_raw else 9999)

            return (grupo, ini, fim)
        else:
            return ('ZZZ', 99999, 99999)  # Para colunas fora do padrão

    # Ordena colunas do tipo (Grupo)X-Y dias
    lista_ordenada = sorted(testeee, key=extrair_ordem)
    resultado = lista_unica2 + lista_ordenada

    # Regras de reorganização
    if 'Concentração Top 10 Cedentes (R$)' in resultado and 'Cedente 1' in resultado:
        resultado.remove('Concentração Top 10 Cedentes (R$)')
        idx = resultado.index('Cedente 1')
        resultado.insert(idx, 'Concentração Top 10 Cedentes (R$)')

    if 'Concentração Top 10 Sacados (R$)' in resultado and 'Sacado 1' in resultado:
        resultado.remove('Concentração Top 10 Sacados (R$)')
        idx = resultado.index('Sacado 1')
        resultado.insert(idx, 'Concentração Top 10 Sacados (R$)')

    padrao_grupo = re.compile(r'^\(([^)]+)\)\s*[\d>]')
    grupos = {}
    for i, col in enumerate(resultado):
        match = padrao_grupo.match(col)
        if match:
            grupo = match.group(1)
            if grupo not in grupos and grupo in resultado:
                grupos[grupo] = i

    for grupo, idx_ref in grupos.items():
        if grupo in resultado:
            resultado.remove(grupo)
            resultado.insert(idx_ref - 1, grupo)

    # Mover 'PDD À Vencer' após último '(PDD Total)X-Y dias'
    if 'PDD À Vencer' in resultado:
        padrao_pdd = re.compile(r'^\(PDD Total\)\s*\d+\s*-\s*\d+\s*dias')
        indices_pdd = [i for i, col in enumerate(resultado) if padrao_pdd.match(col)]
        if indices_pdd:
            idx_destino = max(indices_pdd)
            resultado.remove('PDD À Vencer')
            resultado.insert(idx_destino, 'PDD À Vencer')

    return resultado

In [ ]:
ordem_final = organizar_colunas_df(lista_unica2, testeee)

In [ ]:
# valores dos tipos de ativos financeiros

resultado_quasefinal[ordem_final]

In [ ]:
#checar tipos e informar algumas coisas depois

testeee